# 02 Vector Database

Build and validate the local retrieval layer for the Autonomous Research Assistant. This notebook stores paper metadata in SQLite, indexes processed text chunks in ChromaDB, and checks that semantic retrieval returns useful evidence for downstream RAG generation.

## Setup

The project-root detection keeps imports and file paths stable whether the notebook is launched from the repository root or from the `notebooks/` directory.

In [1]:
from pathlib import Path
import shutil
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.vector_store import SQLiteMetadataStore, ChromaVectorStore, load_embedding_model

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
RAW_CSV = DATA_DIR / 'raw_papers.csv'
PROCESSED_CSV = DATA_DIR / 'processed_papers.csv'
PAPERS_DB = DATA_DIR / 'papers.db'
CHROMA_DIR = DATA_DIR / 'chroma'
RETRIEVAL_EXAMPLES_CSV = OUTPUTS_DIR / 'retrieval_examples.csv'

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 140)
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/Lenovo/Desktop/sem 6/Gen_AI Project


## Load Data

The vector database is built from the processed chunks, while SQLite keeps paper-level metadata.

In [2]:
required_files = [RAW_CSV, PROCESSED_CSV]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required data files: {missing_files}. Run 01_data_collection.ipynb first.')

raw_df = pd.read_csv(RAW_CSV)
processed_df = pd.read_csv(PROCESSED_CSV)

pd.DataFrame(
    {
        'dataset': ['raw_papers', 'processed_chunks'],
        'rows': [len(raw_df), len(processed_df)],
        'columns': [len(raw_df.columns), len(processed_df.columns)],
    }
)

,dataset,rows,columns
0,raw_papers,448,9
1,processed_chunks,928,12


## Data Quality Checks

These checks catch the common causes of weak retrieval: missing text, duplicate chunk IDs, duplicate papers, and topic/source imbalance.

In [3]:
quality_checks = pd.DataFrame(
    {
        'check': [
            'raw missing titles',
            'raw missing abstracts',
            'raw duplicate title+abstract rows',
            'processed missing chunk_text',
            'processed duplicate chunk_id rows',
            'processed chunks without paper_id',
        ],
        'value': [
            int(raw_df['title'].isna().sum()),
            int(raw_df['abstract'].isna().sum()),
            int(raw_df.duplicated(subset=['title', 'abstract']).sum()),
            int(processed_df['chunk_text'].isna().sum()),
            int(processed_df['chunk_id'].duplicated().sum()),
            int(processed_df['paper_id'].isna().sum()),
        ],
    }
)
quality_checks

,check,value
0,raw missing titles,0
1,raw missing abstracts,2
2,raw duplicate title+abstract rows,0
3,processed missing chunk_text,0
4,processed duplicate chunk_id rows,0
5,processed chunks without paper_id,0


In [4]:
topic_source_summary = (
    raw_df.groupby(['topic', 'source'])
    .size()
    .reset_index(name='papers')
    .sort_values(['topic', 'source'])
)

print(f"Unique topics: {raw_df['topic'].nunique()}")
print(f"Sources: {raw_df['source'].value_counts().to_dict()}")
topic_source_summary.head(30)

Unique topics: 24
Sources: {'arxiv': 423, 'semantic_scholar': 25}


,topic,source,papers
0,AI chatbots for education using large language models,arxiv,19
1,AI chatbots for education using large language models,semantic_scholar,25
2,Academic paper recommendation using embeddings,arxiv,24
3,Evaluation of retrieval-augmented generation systems,arxiv,5
4,Hallucination mitigation in retrieval augmented generation,arxiv,22
5,Hugging Face transformer model ecosystem,arxiv,25
6,Instruction-tuned open-source language models,arxiv,21
7,LLM hallucination detection and mitigation,arxiv,25
8,Large language models for literature review generation,arxiv,8
9,Large language models for question answering over research papers,arxiv,18


## Rebuild Metadata Store

The SQLite database stores paper-level metadata and can be joined back to retrieved chunk IDs for inspection and reporting.

In [5]:
REBUILD_INDEX = True

if REBUILD_INDEX and PAPERS_DB.exists():
    PAPERS_DB.unlink()

metadata_store = SQLiteMetadataStore(str(PAPERS_DB))
metadata_store.upsert_papers(raw_df)
metadata_df = metadata_store.fetch_all()

pd.DataFrame(
    {
        'store': ['SQLite papers table'],
        'rows': [len(metadata_df)],
        'expected_rows': [raw_df['paper_id'].nunique()],
        'matches_expected': [len(metadata_df) == raw_df['paper_id'].nunique()],
    }
)

,store,rows,expected_rows,matches_expected
0,SQLite papers table,448,448,True


In [6]:
metadata_df[['paper_id', 'title', 'year', 'venue', 'topic', 'source']].head(10)

,paper_id,title,year,venue,topic,source
0,2409.11272v7,LOLA -- An Open-Source Massively Multilingual Large Language Model,2024,arXiv,Open-source large language models,arxiv
1,2501.05032v2,Enhancing Human-Like Responses in Large Language Models,2025,arXiv,Open-source large language models,arxiv
2,2503.16581v1,Investigating Retrieval-Augmented Generation in Quranic Studies: A Study of 13 Open-Source Large Language Models,2025,arXiv,Open-source large language models,arxiv
3,2402.14679v2,Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality,2024,arXiv,Open-source large language models,arxiv
4,2309.13734v2,Prompting and Fine-Tuning Open-Sourced Large Language Models for Stance Classification,2023,arXiv,Open-source large language models,arxiv
5,2405.11357v3,Large Language Models Lack Understanding of Character Composition of Words,2024,arXiv,Open-source large language models,arxiv
6,2403.09676v1,Unmasking the Shadows of AI: Investigating Deceptive Capabilities in Large Language Models,2024,arXiv,Open-source large language models,arxiv
7,2310.00905v2,All Languages Matter: On the Multilingual Safety of Large Language Models,2023,arXiv,Open-source large language models,arxiv
8,2204.06745v1,GPT-NeoX-20B: An Open-Source Autoregressive Language Model,2022,arXiv,Open-source large language models,arxiv
9,2407.01505v1,Self-Cognition in Large Language Models: An Exploratory Study,2024,arXiv,Open-source large language models,arxiv


## Build Chroma Vector Index

Chroma stores chunk embeddings for semantic search. Rebuilding the directory avoids stale vectors when the dataset changes.

In [7]:
if REBUILD_INDEX and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

embedding_model = load_embedding_model()
vector_store = ChromaVectorStore(str(CHROMA_DIR))
vector_store.index_chunks(processed_df, embedding_model)

chroma_count = vector_store.collection.count()
pd.DataFrame(
    {
        'store': ['Chroma research_papers collection'],
        'indexed_chunks': [chroma_count],
        'expected_chunks': [len(processed_df)],
        'matches_expected': [chroma_count == len(processed_df)],
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,store,indexed_chunks,expected_chunks,matches_expected
0,Chroma research_papers collection,928,928,True


## Retrieval Helper

This helper returns retrieved documents, metadata, and Chroma distances as a table. Lower distance generally means a closer match.

In [8]:
def search_to_dataframe(query: str, top_k: int = 5) -> pd.DataFrame:
    query_embedding = embedding_model.encode([query]).tolist()
    results = vector_store.collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=['documents', 'metadatas', 'distances'],
    )
    rows = []
    documents = results.get('documents', [[]])[0]
    metadatas = results.get('metadatas', [[]])[0]
    distances = results.get('distances', [[]])[0]
    ids = results.get('ids', [[]])[0]

    for rank, (chunk_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances), start=1):
        rows.append(
            {
                'query': query,
                'rank': rank,
                'distance': distance,
                'chunk_id': chunk_id,
                'paper_id': metadata.get('paper_id'),
                'title': metadata.get('title'),
                'topic': metadata.get('topic'),
                'source': metadata.get('source'),
                'url': metadata.get('url'),
                'chunk_text': document,
            }
        )
    return pd.DataFrame(rows)

search_to_dataframe('citation grounded literature review generation', top_k=5)

,query,rank,distance,chunk_id,paper_id,title,topic,source,url,chunk_text
0,citation grounded literature review generation,1,0.740801,2308.02443v1_chunk_1,2308.02443v1,AI Literature Review Suite,Large language models for literature review generation,arxiv,http://arxiv.org/abs/2308.02443v1,"(GUI). The suite also features integrated programs for bibliographic organization, interaction and query, and literature review summarie..."
1,citation grounded literature review generation,2,0.755942,2411.18583v1_chunk_0,2411.18583v1,Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation,Large language models for literature review generation,arxiv,http://arxiv.org/abs/2411.18583v1,Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation. This research presents and compares multi...
2,citation grounded literature review generation,3,0.871430,2308.02443v1_chunk_0,2308.02443v1,AI Literature Review Suite,Large language models for literature review generation,arxiv,http://arxiv.org/abs/2308.02443v1,AI Literature Review Suite. The process of conducting literature reviews is often time-consuming and labor-intensive. To streamline this...
3,citation grounded literature review generation,4,0.945495,2603.20235v1_chunk_0,2603.20235v1,"Writing literature reviews with AI: principles, hurdles and some lessons learned",RAG for literature review synthesis,arxiv,http://arxiv.org/abs/2603.20235v1,"Writing literature reviews with AI: principles, hurdles and some lessons learned. We qualitatively compared literature reviews produced ..."
4,citation grounded literature review generation,5,1.052860,2508.05666v1_chunk_0,2508.05666v1,HySemRAG: A Hybrid Semantic Retrieval-Augmented Generation Framework for Automated Literature Synthesis and Methodological Gap Analysis,RAG for literature review synthesis,arxiv,http://arxiv.org/abs/2508.05666v1,HySemRAG: A Hybrid Semantic Retrieval-Augmented Generation Framework for Automated Literature Synthesis and Methodological Gap Analysis....


## Retrieval Test Queries

These queries cover the main use cases of the app: literature review synthesis, research gap analysis, hallucination control, and technical explanation.

In [9]:
test_queries = [
    'citation grounded literature review generation',
    'research gaps in RAG systems for academic question answering',
    'hallucination mitigation for retrieval augmented generation',
    'LoRA fine tuning for open source language models',
    'semantic search for scholarly paper recommendation',
]

retrieval_examples_df = pd.concat(
    [search_to_dataframe(query, top_k=5) for query in test_queries],
    ignore_index=True,
)
retrieval_examples_df.to_csv(RETRIEVAL_EXAMPLES_CSV, index=False)
retrieval_examples_df[['query', 'rank', 'distance', 'title', 'topic', 'source']].head(25)

,query,rank,distance,title,topic,source
0,citation grounded literature review generation,1,0.740801,AI Literature Review Suite,Large language models for literature review generation,arxiv
1,citation grounded literature review generation,2,0.755942,Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation,Large language models for literature review generation,arxiv
2,citation grounded literature review generation,3,0.871430,AI Literature Review Suite,Large language models for literature review generation,arxiv
3,citation grounded literature review generation,4,0.945495,"Writing literature reviews with AI: principles, hurdles and some lessons learned",RAG for literature review synthesis,arxiv
4,citation grounded literature review generation,5,1.052860,HySemRAG: A Hybrid Semantic Retrieval-Augmented Generation Framework for Automated Literature Synthesis and Methodological Gap Analysis,RAG for literature review synthesis,arxiv
5,research gaps in RAG systems for academic question answering,1,0.704269,RAG-based Question Answering over Heterogeneous Data and Text,RAG for citation-grounded text generation,arxiv
6,research gaps in RAG systems for academic question answering,2,0.719676,Fine-tune the Entire RAG Architecture (including DPR retriever) for Question-Answering,RAG for citation-grounded text generation,arxiv
7,research gaps in RAG systems for academic question answering,3,0.827180,SF-RAG: Structure-Fidelity Retrieval-Augmented Generation for Academic Question Answering,Retrieval-augmented generation for academic question answering,arxiv
8,research gaps in RAG systems for academic question answering,4,0.837205,MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries,RAG for literature review synthesis,arxiv
9,research gaps in RAG systems for academic question answering,5,0.849717,FARSIQA: Faithful and Advanced RAG System for Islamic Question Answering,Large language models for question answering over research papers,arxiv


## Retrieval Diagnostics

These diagnostics are not a full benchmark, but they show whether retrieval is diverse enough to support grounded generation.

In [10]:
retrieval_diagnostics = (
    retrieval_examples_df.groupby('query')
    .agg(
        retrieved_chunks=('chunk_id', 'count'),
        unique_papers=('paper_id', 'nunique'),
        unique_topics=('topic', 'nunique'),
        min_distance=('distance', 'min'),
        mean_distance=('distance', 'mean'),
    )
    .reset_index()
)
retrieval_diagnostics

,query,retrieved_chunks,unique_papers,unique_topics,min_distance,mean_distance
0,LoRA fine tuning for open source language models,5,5,2,0.638628,0.698656
1,citation grounded literature review generation,5,4,2,0.740801,0.873305
2,hallucination mitigation for retrieval augmented generation,5,5,2,0.723085,0.763792
3,research gaps in RAG systems for academic question answering,5,5,4,0.704269,0.787609
4,semantic search for scholarly paper recommendation,5,4,2,0.569413,0.696541


In [11]:
duplicate_retrievals = (
    retrieval_examples_df.groupby(['query', 'paper_id'])
    .size()
    .reset_index(name='retrieved_chunks_from_same_paper')
    .query('retrieved_chunks_from_same_paper > 1')
    .sort_values(['query', 'retrieved_chunks_from_same_paper'], ascending=[True, False])
)
duplicate_retrievals

,query,paper_id,retrieved_chunks_from_same_paper
5,citation grounded literature review generation,2308.02443v1,2
22,semantic search for scholarly paper recommendation,2601.19513v1,2


## Final Artifact Check

The generated retrieval examples can be used directly in the final report for evidence retrieval screenshots or tables.

In [12]:
artifact_paths = [PAPERS_DB, CHROMA_DIR / 'chroma.sqlite3', RETRIEVAL_EXAMPLES_CSV]
pd.DataFrame(
    {
        'artifact': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths],
        'exists': [path.exists() for path in artifact_paths],
        'size_bytes': [path.stat().st_size if path.is_file() else 0 for path in artifact_paths],
    }
)

,artifact,exists,size_bytes
0,data/papers.db,True,860160
1,data/chroma/chroma.sqlite3,True,9904128
2,outputs/retrieval_examples.csv,True,26980
